In [ ]:
import torch
from datasets import Dataset
import math
from transformers import (
    AutoTokenizer,
    AutoConfig,
    LlamaForCausalLM,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

model_id = './model'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# from datasets import Dataset
# import random
#
# SEP = "<|eos|>"
#
# with open("./data.txt", "r", encoding="utf-8") as f:
#     docs = [x.strip() for x in f.read().split(SEP)]
#
# MIN_DOC_CHARS = 200  # or >= 400
# docs = [d for d in docs if d and len(d) >= MIN_DOC_CHARS]
#
# # docs = docs * 2
#
# random.shuffle(docs)
#
# raw_datasets = Dataset.from_dict({"text": docs}).train_test_split(test_size=0.1)
# print(raw_datasets)


In [ ]:
from datasets import Dataset

DATA_FILE = "./data.txt"
SEP = "<|eos|>"
MIN_DOC_CHARS = 200

def iter_docs(path):
    acc = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line == SEP:
                doc = "\n".join(acc).strip()
                acc = []
                if doc and len(doc) >= MIN_DOC_CHARS:
                    yield {"text": doc}
            else:
                acc.append(line)

        if acc:
            doc = "\n".join(acc).strip()
            if doc and len(doc) >= MIN_DOC_CHARS:
                yield {"text": doc}

dataset = Dataset.from_generator(iter_docs, gen_kwargs={"path": DATA_FILE})
dataset = dataset.shuffle(seed=42)
raw_datasets = dataset.train_test_split(test_size=0.01, seed=42)

print(raw_datasets)

In [ ]:
# from transformers import AutoTokenizer
# from datasets import Dataset
# import numpy as np
#
# tokenizer = AutoTokenizer.from_pretrained("./model")
#
# def count_tokens(batch):
#     enc = tokenizer(batch["text"], truncation=False, padding=False)
#     return {
#         "n_tokens": [len(x) + 1 for x in enc["input_ids"]]  # +1 for eos
#     }
#
# token_stats = dataset.map(
#     count_tokens,
#     batched=True,
#     batch_size=1000,
#     num_proc=4,
# )
#
# total_tokens = int(sum(token_stats["n_tokens"]))
# avg_tokens = float(np.mean(token_stats["n_tokens"]))
# median_tokens = float(np.median(token_stats["n_tokens"]))
#
# print("docs:", len(dataset))
# print("total_tokens:", total_tokens)
# print("avg_tokens:", avg_tokens)
# print("median_tokens:", median_tokens)

In [ ]:
block_size = 512
# stride = block_size // 2
stride = block_size

def tokenize(examples):
    texts = [t + tokenizer.eos_token for t in examples["text"]]
    return tokenizer(texts, truncation=False, padding=False)

tokenized = raw_datasets.map(
    tokenize,
    batched=True,
    num_proc=4,
    remove_columns=raw_datasets["train"].column_names,
)

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    if total_length < block_size:
        return {"input_ids": [], "labels": []}

    # sliding window
    input_ids = []
    for i in range(0, total_length - block_size + 1, stride):
        input_ids.append(concatenated["input_ids"][i:i+block_size])

    return {"input_ids": input_ids, "labels": [x[:] for x in input_ids]}

lm_datasets = tokenized.map(
    group_texts,
    batched=True,
    num_proc=4,
    remove_columns=tokenized["train"].column_names,
)

lm_datasets.set_format(type="torch")
lm_datasets

In [ ]:
print(tokenizer.decode(lm_datasets['test'][0]['input_ids']))
print('---')
print(tokenizer.decode(lm_datasets['test'][1]['input_ids']))

In [ ]:
# config = AutoConfig.from_pretrained(
#     pretrained_model_name_or_path='HuggingFaceTB/SmolLM2-135M',
#     vocab_size=len(tokenizer),
#     bos_token_id=tokenizer.eos_token_id,
#     eos_token_id=tokenizer.eos_token_id,
#     pad_token_id=tokenizer.pad_token_id,
#     hidden_size=384,
#     intermediate_size=384*4,
#     num_attention_heads=6,
#     num_hidden_layers=8,
#     num_key_value_heads=3,
#     max_position_embeddings=block_size * 2,
# )
# model = LlamaForCausalLM(config)

from transformers import LlamaConfig, LlamaForCausalLM

config = LlamaConfig(
    vocab_size=len(tokenizer),
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,

    hidden_size=576,
    intermediate_size=4 * 576,  # 4x hidden
    num_attention_heads=9,
    num_hidden_layers=10,
    num_key_value_heads=3,

    max_position_embeddings=block_size,
    rms_norm_eps=1e-5,
    tie_word_embeddings=True,
)
model = LlamaForCausalLM(config)
model_size = sum(t.numel() for t in model.parameters())
print(f"Transformer size: {model_size/1000**2:.1f}M parameters")

In [ ]:
training_args = TrainingArguments(
    output_dir=model_id,
    overwrite_output_dir=True,
    eval_strategy="steps",
    eval_steps=5000,
    logging_steps=5000,
    save_steps=5000,
    save_total_limit=5,
    num_train_epochs=5,      # bump later
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2, # 4 for mid
    learning_rate=3e-4,
    warmup_ratio=0.01,
    weight_decay=0.0,
    fp16=True,
    bf16=False,
    report_to=[],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

class PerplexitySFTTrainer(Trainer):
    def log(self, logs, *args, **kwargs):
        if "eval_loss" in logs:
            logs["eval_perplexity"] = round(math.exp(logs["eval_loss"]), 2)
        super().log(logs, *args, **kwargs)

trainer = PerplexitySFTTrainer(
    processing_class=tokenizer,
    model=model,
    args=training_args,
    train_dataset=lm_datasets['train'],
    eval_dataset=lm_datasets['test'],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(model_id)

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)

In [ ]:
import matplotlib.pyplot as plt
logs = trainer.state.log_history

train_steps = []
train_losses = []

eval_steps = []
eval_losses = []

for log in logs:
    if 'loss' in log and 'eval_loss' not in log:
        train_steps.append(log['step'])
        train_losses.append(log['loss'])

    if 'eval_loss' in log:
        eval_steps.append(log['step'])
        eval_losses.append(log['eval_loss'])


plt.figure(figsize=(12, 6))

plt.plot(train_steps, train_losses, label="Training Loss", linewidth=2)
plt.plot(eval_steps, eval_losses, label="Validation Loss", linewidth=2)

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)

plt.show()